# Voice AI- Give Your AI Ears and a Voice

**Module 6 · Lesson 6.7**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Voice Loop - Listen, Think, Speak
- Speech-to-Text - How a Machine Hears
- Using STT - Transcribe and Understand
- Text-to-Speech - Giving It a Voice
- Building a Voice Agent - Cascaded vs Speech-to-Speech
- The 2026 Landscape, Choosing and Responsible Use

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

## The whole game: an AI that listens and talks back

> 🗣️ **Analogy**
>
> **Picture a great receptionist.** A guest says "I need a taxi to the airport at 6am." The receptionist **listens** and gets the words (Speech-to-Text), **thinks** - checks the schedule, books a car (the LLM from Module 5) - and **speaks** back in a natural voice: "Done, your taxi is booked for 6am" (Text-to-Speech). That listen-think-speak loop is every voice assistant: Alexa, Siri, a call-centre bot.
> Everything in this lesson is that one loop: **audio in, audio out, with an LLM in the middle**. Transcribe a meeting, dictate a message, run a voice agent - all the same three parts, each a sequence model you already have the tools to understand.
> **Where the analogy breaks down:** a human receptionist hears tone, interruptions, and mixed languages effortlessly and replies instantly. A voice AI pays a *latency tax* at every step (detect speech, transcribe, think, synthesize) and can mishear accents and noise - so real-time voice is a streaming and turn-taking problem, not just three good models bolted together.

This is the **close of Module 6**: the last modality. It builds on Lesson 6.5 (multimodal models - audio can go straight into an LLM), reuses the LLM and prompting from Module 5 (the "think" step), and carries forward 6.6's deepfake/provenance thread (now for voice). Two forward hooks: production voice agents that orchestrate this pipeline live in Module 11, and the responsible-use depth - voice deepfakes, consent, watermarking - is Module 15.

- **Explain** the voice loop (STT -> LLM -> TTS) and how each half works - STT turns audio into a spectrogram then text with an encoder-decoder; TTS turns text into a waveform via an acoustic model plus a vocoder, with voice cloning as conditioning on a short reference.

- **Apply** modern STT (Whisper locally, a hosted streaming API, and Gemini native-audio understanding) and TTS (an open model plus a cloning option), including timestamps, streaming, and prosody/SSML.

- **Analyze** the voice-agent tradeoff - cascaded (STT -> LLM -> TTS: auditable, swappable) vs speech-to-speech (lower latency, natural, but a black box) - and the latency/turn-taking budget.

- **Evaluate** the 2026 landscape (open vs hosted STT/TTS, WER/MOS, cost, latency) and the responsible-use stack (voice as biometric, consent, deepfakes, watermarking) that Module 15 develops.

## Warm-up: 60 seconds of recall

Tap each card to check yourself. Each one is load-bearing for today.

---

## 🎯 Concept 1: The Voice Loop - Listen, Think, Speak

### The Voice Loop - Listen, Think, Speak

Three parts in a line: Speech-to-Text, an LLM, Text-to-Speech. The delay is the sum of all three.

**A translator on a phone call.** One person speaks; the translator writes down what was said (STT), decides how to answer (the LLM), then says the answer out loud (TTS). Three jobs, one after another. The whole call feels slow if any one of them dawdles - because you wait for all three.

The gain: a voice loop = STT + LLM + TTS. Total delay is the sum of the parts, so real-time voice is about streaming, not one fast model.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  A voice AI is one loop: **Speech-to-Text** turns your voice into words, an **LLM** reads them and decides a reply, and **Text-to-Speech** turns that reply into a natural voice. The whole thing feels responsive only if the total delay - the *sum* of all four stages (detect, transcribe, think, speak) - stays under about a second, which is why streaming and turn-taking (Step 5) matter as much as the models.

---

## 🎯 Concept 2: Speech-to-Text - How a Machine Hears

### Speech-to-Text - How a Machine Hears

Audio becomes a spectrogram, an encoder-decoder reads it, and out comes text - in context.

**Reading a heart-rate monitor.** The raw squiggle is hard to read directly, so you turn it into a clearer chart first, then read the pattern. STT does the same: it turns the sound wave into a *spectrogram* (a chart of pitches over time), then a model reads the chart and writes the words - using the whole sentence for context, like you would.

The gain: STT = waveform -> spectrogram -> encoder-decoder -> text. Context is why it beats letter-by-letter guessing; accent and noise are the hard part.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Speech-to-Text turns a **waveform** into a **spectrogram** (a picture of frequencies over time), then an **encoder-decoder transformer** (Whisper) reads that picture and emits text tokens. Because it uses whole-sentence context, it resolves homophones and punctuation - and struggles with accents, noise, and code-switching, not single sounds. **Batch** STT transcribes a finished clip; **streaming** STT emits partial words live (Step 3).

---

## 🎯 Concept 3: Using STT - Transcribe and Understand

### Using STT - Transcribe and Understand

Whisper locally, a hosted API for streaming, and an LLM that understands audio directly.

**A typist vs a live captioner.** A typist takes your finished voicemail and types it up (batch - great for files). A live captioner types as you speak, showing best-guess words that firm up a moment later (streaming - what a voice agent needs). Same skill, different delivery.

The gain: Whisper (open, batch) for files; a streaming API for live; or send audio straight to an LLM for understanding. Match the tool to batch vs real-time.

**📝 `transcribe.py Whisper +`** - *google-genai*

In [ ]:
# pip install openai-whisper google-genai
# --- 1) Whisper: open, local, batch (great for files) ---
import whisper

model = whisper.load_model("large-v3-turbo")   # turbo: ~5x faster decoder, near-v3 accuracy
result = model.transcribe("meeting.mp3")        # transcribes the WHOLE clip, then returns
print(result["text"][:54], "...")
print("detected language:", result["language"])
for seg in result["segments"][:1]:      # timestamps, e.g. for subtitles
    print(f"[{seg['start']:.1f}s-{seg['end']:.1f}s] {seg['text'].strip()}")
# Output: Welcome everyone, let's get started with the quarterly ...
# Output: detected language: en
# Output: [0.0s-3.4s] Welcome everyone, let's get started with the quarterly review.

# --- 2) Gemini: send audio straight to an LLM (transcribe AND understand) ---
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
with open("question.wav", "rb") as f:
    audio = f.read()

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["Transcribe this, then answer the question it asks.",
              types.Part.from_bytes(data=audio, mime_type="audio/wav")],
)
print(resp.text)
# Output: You asked "what's the capital of Japan?" - it is Tokyo.

- **Whisper is batch:** `model.transcribe()` reads a whole file and returns text, the detected language, and timestamped `segments`. `large-v3-turbo` shrinks the decoder for ~5x speed at near-v3 accuracy; `faster-whisper` (CTranslate2) goes faster still, and `whisper.cpp` runs on a laptop CPU.

- **Native audio understanding:** the unified **google-genai** SDK sends audio as a `types.Part` straight into `gemini-3-flash` - it transcribes *and* reasons (answers, summarizes, translates) in one call. This is the 6.5 "native multimodal" idea, now for sound; the older `google.generativeai` was deprecated in 2025.

- **For live audio, reach for streaming:** Whisper waits for the whole clip, so real-time agents use a streaming API - **Deepgram** (Nova-3, sub-300ms; Flux adds built-in end-of-turn detection) or **AssemblyAI** - that emits partial words as you speak. Same idea, delivered live.

#### 💡 What just happened

⚡ What just happened?
  Three ways to get text from speech: **Whisper** (open, local, batch - files, timestamps, language detection), a **streaming API** (Deepgram/AssemblyAI - live partials for voice agents), and a **native-audio LLM** (`gemini-3-flash` via google-genai - transcribe and understand in one call). Whisper is the privacy/cost default for files; streaming APIs own real-time; the LLM path wins when you want understanding, not just a transcript.

---

## 🎯 Concept 4: Text-to-Speech - Giving It a Voice

### Text-to-Speech - Giving It a Voice

Text becomes a waveform via an acoustic model and a vocoder - and cloning needs no training.

**A voice actor reading a script.** Give them words and they produce smooth, natural speech - not clipped syllables. For cloning, think of a skilled impressionist: hand them a 5-second sample of someone's voice and they can say *anything* in that voice, without "studying" for weeks. The model does the same by conditioning on the sample, not training on it.

The gain: TTS = text -> acoustic model -> vocoder -> waveform; cloning conditions on a short reference. Natural flow, and no per-voice training for cloning.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**📝 `text_to_speech.py Kokoro`** - *(open)*

In [ ]:
# pip install kokoro soundfile   -  open TTS (Apache-2.0), runs on CPU, 82M params
from kokoro import KPipeline
import soundfile as sf

pipeline = KPipeline(lang_code="a")   # 'a' = American English (preset voices, no cloning)
text = "Welcome to the course. Your voice AI journey starts here."

# The pipeline yields (graphemes, phonemes, audio) chunks; write each to a wav.
for i, (graphemes, phonemes, audio) in enumerate(pipeline(text, voice="af_heart")):
    sf.write(f"line_{i}.wav", audio, 24000)   # 24 kHz waveform
print("saved line_0.wav (24 kHz)")
# Output: saved line_0.wav (24 kHz)

- **Kokoro** is the open, CPU-friendly default: 82M params, Apache-2.0, 54 preset voices, *no* cloning. `KPipeline` turns text into phonemes (the individual speech sounds a word is made of, like /k/ /a/ /t/ for "cat") then a 24 kHz waveform - the acoustic-model-plus-vocoder pipeline from the animation, in one call.

- **Prosody / SSML:** hosted engines (Google, Azure, ElevenLabs) accept **SSML** to control delivery - `<break time="400ms"/>` for a pause, `<emphasis>` to stress a word, `<prosody rate="slow" pitch="+2st">` for speed and pitch. Same words, different feeling.

- **Voice cloning is a separate capability:** for zero-shot cloning from a few seconds, use **F5-TTS** (open, flow matching) or a hosted service like **ElevenLabs**. It conditions on a reference clip - it does not train on the voice - which is exactly why consent matters (Step 6).

- **Quality is close, license decides:** the best open TTS (Sesame CSM, ~4.7 MOS - Mean Opinion Score, a 1-to-5 human naturalness rating where higher is better) is within ~0.1-0.3 MOS of ElevenLabs, so pick on license (Kokoro Apache vs F5 non-commercial), cloning need, languages, and CPU-vs-GPU - not naturalness alone.

#### 💡 What just happened

⚡ What just happened?
  Text-to-Speech turns words into a **continuous waveform** via an **acoustic model** (text -> a spectrogram-like representation) plus a **vocoder** (that -> audio), so it flows naturally instead of stitching clips. **SSML** controls delivery (pauses, emphasis, pitch), and **zero-shot cloning** (F5-TTS, ElevenLabs) copies a voice by *conditioning* on a short reference clip - no training - which is powerful and a deepfake risk.

---

## 🎯 Concept 5: Building a Voice Agent - Cascaded vs Speech-to-Speech

### Building a Voice Agent - Cascaded vs Speech-to-Speech

Wire STT, an LLM, and TTS into a loop - or use one speech-to-speech model. It is a latency-vs-visibility choice.

**A translator with a notepad vs a bilingual friend.** The notepad translator writes down each step (you can read what they heard and decided - slower, but you can check their work). The bilingual friend just answers instantly in the other language (fast and natural, but you cannot see how they got there). Cascaded is the notepad; speech-to-speech is the friend.

The gain: cascaded (STT->LLM->TTS) = visible and swappable; speech-to-speech = faster and more natural but opaque. Regulated work stays cascaded; frameworks handle the plumbing.

**📝 `voice_agent.py Cascaded`** - *pipeline*

In [ ]:
# A cascaded voice agent: STT -> LLM -> TTS. Frameworks (LiveKit, Pipecat)
# handle the mic, WebRTC, and turn-taking; here is the core loop, decomposed.
import os, whisper, soundfile as sf
import numpy as np
from google import genai
from kokoro import KPipeline

class VoiceAgent:
    def __init__(self):
        self.stt = whisper.load_model("large-v3-turbo")          # 1. ears
        self.llm = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])  # 2. brain
        self.tts = KPipeline(lang_code="a")                         # 3. voice

    def transcribe(self, wav_path: str) -> str:
        return self.stt.transcribe(wav_path)["text"].strip()   # you can LOG this

    def think(self, user_text: str) -> str:
        reply = self.llm.models.generate_content(
            model="gemini-3-flash",
            contents=[f"Reply in one short spoken sentence: {user_text}"],
        )
        return reply.text.strip()                            # and LOG this

    def speak(self, text: str, out="reply.wav"):
        # Kokoro yields ONE chunk per sentence - concatenate them all, then
        # write ONE file. A naive sf.write() inside the loop would overwrite
        # the file each pass and keep only the last chunk.
        parts = [audio for _, _, audio in self.tts(text, voice="af_heart")]
        sf.write(out, np.concatenate(parts), 24000)
        return out

    def turn(self, wav_path: str) -> str:
        heard = self.transcribe(wav_path)                    # STT
        reply = self.think(heard)                            # LLM
        print(f"heard: {heard}\nreply: {reply}")
        return self.speak(reply)                             # TTS

agent = VoiceAgent()
agent.turn("user_question.wav")
# Output: heard: what is the capital of France
# Output: reply: The capital of France is Paris.

- **Cascaded = three visible stages:** `transcribe` (STT), `think` (the LLM), `speak` (TTS). The key production property is that you can **log the transcript and the reply**, redact sensitive text between stages, and swap any stage (a different STT or voice) without touching the others.

- **Frameworks do the hard real-time parts:** **LiveKit Agents** and **Pipecat** (both open) handle the microphone, WebRTC transport, voice-activity detection, and *turn detection* (knowing when you stopped talking) - the plumbing that makes it feel live. This snippet is the core loop they wrap.

- **Speech-to-speech is the alternative:** the **OpenAI Realtime API** and **Gemini Live** take audio in and emit audio out with lower latency and more natural prosody - but no transcript to inspect. Reach for it when latency and naturalness beat auditability.

- **Latency is a budget:** voice-to-voice delay is VAD + STT + LLM + TTS; you hit sub-second only by *streaming* at every boundary, not by picking one fast model.

#### 💡 What just happened

⚡ What just happened?
  A voice agent is the loop made real. **Cascaded** (STT -> LLM -> TTS) is the production default - auditable, redactable, swappable - and open frameworks (**LiveKit**, **Pipecat**) handle the mic, WebRTC, VAD, and turn-taking. **Speech-to-speech** (OpenAI Realtime, Gemini Live) is faster and more natural but a black box. The delay is a budget you pay across every stage, so you stream at each boundary to feel real-time.

---

## 🎯 Concept 6: The 2026 Landscape, Choosing and Responsible Use

### The 2026 Landscape, Choosing and Responsible Use

Pick STT/TTS by constraint - and remember a voice is biometric.

**Buying shoes for a specific trip.** You do not buy "the best shoe"; you buy for the terrain - trail, road, formal. Voice tools are the same: pick for real-time-vs-batch, private-vs-hosted, cloning-or-not, and which languages. And whatever you pick, a voice is like a fingerprint - you handle it with consent and care.

The gain: choose by constraint, not benchmark; treat voice as biometric. Consent and watermarking ship with any voice product.

| Need | Reach for (as of mid-2026) | Why |
|---|---|---|
| Transcribe files, private/free | Whisper large-v3-turbo (open) | Local, timestamps, ~99 languages; faster-whisper / whisper.cpp for speed |
| Real-time voice agent STT | Deepgram Nova-3 / Flux | Sub-300ms streaming + built-in end-of-turn detection |
| Transcribe + understand in one call | Gemini native audio | Audio straight into the LLM; answer/summarize/translate |
| TTS, open and CPU-friendly | Kokoro (82M, Apache-2.0) | Runs on CPU, 54 preset voices, commercial-friendly license |
| TTS with voice cloning | F5-TTS (open, non-commercial) or ElevenLabs (hosted) | Zero-shot cloning from a few seconds; check the license |
| Production voice-agent plumbing | LiveKit / Pipecat (open) | Mic, WebRTC, VAD, turn detection; cascaded or hybrid |

**📝 `choose_voice_stack.py`** - *Decision*

In [ ]:
# Pick voice tools by the constraint that binds - not by one benchmark.
def choose_voice_stack(need: str) -> str:
    table = {
        "files_private":  "Whisper large-v3-turbo (open, batch, timestamps)",
        "realtime_stt":   "Deepgram Nova-3 / Flux (streaming, end-of-turn)",
        "understand":     "Gemini native audio (transcribe + reason in one call)",
        "tts_open":       "Kokoro (82M, Apache-2.0, runs on CPU)",
        "tts_cloning":    "F5-TTS (open) or ElevenLabs (hosted) - mind the license",
    }
    return table.get(need, "start open + cascaded; add hosted/streaming when latency or scale forces it")

for need in ["files_private", "realtime_stt", "tts_open", "tts_cloning"]:
    print(f"{need:14} -> {choose_voice_stack(need)}")
# Output: files_private  -> Whisper large-v3-turbo (open, batch, timestamps)
# Output: realtime_stt   -> Deepgram Nova-3 / Flux (streaming, end-of-turn)
# Output: tts_open       -> Kokoro (82M, Apache-2.0, runs on CPU)
# Output: tts_cloning    -> F5-TTS (open) or ElevenLabs (hosted) - mind the license

- **Constraint, not benchmark:** real-time-vs-batch, private-vs-hosted, cloning-or-not, and languages pick the tool. WER (STT) and MOS (TTS) are close at the top, so they rarely decide alone.

- **Open vs hosted:** open (Whisper, Kokoro, F5, LiveKit) wins on privacy and cost; hosted (Deepgram, ElevenLabs, Gemini) wins on streaming, latency, and zero ops. Native-audio LLMs win when you want understanding, not just text.

- **The numbers move:** models, latency, and prices are mid-2026 and change monthly - treat this as a decision procedure and re-check current models before you commit.

> 📦 **Info**
>
> ⚠️ The responsibility that ships with a voice
> A voice is **biometric** - as identifying as a fingerprint - and zero-shot cloning makes convincing **voice deepfakes** cheap (vishing scams, impersonation). So responsible voice AI is part of the build: get **explicit consent** before cloning a voice, **disclose** AI-generated speech, and **watermark** output (for example Meta's **AudioSeal**, embedded at generation time and detectable after). The deep treatment - consent law, detection, provenance, and the regulatory landscape - is **Module 15 (Ethics & Responsible AI)**; here, just internalize that shipping a synthetic voice means shipping consent and provenance with it.

#### 💡 What just happened

⚡ What just happened?
  Choose voice tools by the **constraint that binds** - real-time vs batch, private vs hosted, cloning or not, which languages - because WER and MOS are close at the top. Open tools (Whisper, Kokoro, F5, LiveKit) win on privacy and cost; hosted tools (Deepgram, ElevenLabs, Gemini) win on streaming and zero ops. And a voice is **biometric**, so consent, disclosure, and watermarking (AudioSeal) ship with any voice product - the responsibility Module 15 develops.

## Synthesis: ears and a voice, closing the module

### The complete picture, in one breath

Voice AI is one loop - **Speech-to-Text -> an LLM -> Text-to-Speech**. STT turns a **waveform into a spectrogram into text** with an encoder-decoder (Whisper), batch or streaming; TTS turns **text into a waveform** with an acoustic model plus a vocoder, and **zero-shot cloning** copies a voice by conditioning on a short reference, no training. A real agent is **cascaded** (auditable, swappable - LiveKit/Pipecat) or **speech-to-speech** (faster, but a black box), and its responsiveness is a **latency budget** paid across every stage. You choose by the constraint that binds, and because a voice is **biometric**, consent and watermarking ship with it. That closes Module 6: the same generative and transformer machinery, now across images, video, 3D, and sound.

> ℹ️ **Info**
>
> Where this goes next
> - Production **voice agents** - orchestrating STT -> LLM -> TTS with tools, memory, and interruption handling (LiveKit/Pipecat at scale) - are developed in Module 11 (AI Agents).
> - The responsible-use stack for voice - deepfakes, consent, disclosure, watermarking (AudioSeal, C2PA), and detection - gets its full treatment in Module 15 (Ethics & Responsible AI).
> - You have now closed the modalities of Module 6: images (6.1-6.4), multimodal understanding (6.5), video and 3D generation (6.6), and voice (6.7).

- Radford et al., *Robust Speech Recognition via Large-Scale Weak Supervision (Whisper)* (2022) - [arxiv.org/abs/2212.04356](https://arxiv.org/abs/2212.04356)

- Baevski et al., *wav2vec 2.0* (2020) - [arxiv.org/abs/2006.11477](https://arxiv.org/abs/2006.11477) (self-supervised speech representations)

- Wang et al., *Neural Codec Language Models are Zero-Shot TTS (VALL-E)* (2023) - [arxiv.org/abs/2301.02111](https://arxiv.org/abs/2301.02111) (in-context voice cloning)

- Chen et al., *F5-TTS: Flow Matching for Fluent and Faithful Speech* (2024) - [arxiv.org/abs/2410.06885](https://arxiv.org/abs/2410.06885)

- Defossez et al., *High Fidelity Neural Audio Compression (EnCodec)* (2022) - [arxiv.org/abs/2210.13438](https://arxiv.org/abs/2210.13438) (the audio codec that tokenizes speech)

- Kim et al., *VITS: End-to-End Text-to-Speech* (2021) - [arxiv.org/abs/2106.06103](https://arxiv.org/abs/2106.06103) · Li et al., *StyleTTS 2* (2023) - [arxiv.org/abs/2306.07691](https://arxiv.org/abs/2306.07691) (behind Kokoro-class TTS)

- OpenAI Whisper - [github.com/openai/whisper](https://github.com/openai/whisper) · Kokoro-82M - [huggingface.co/hexgrad/Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M) · LiveKit Agents - [github.com/livekit/agents](https://github.com/livekit/agents)

## Recap

> ✅ **Info**
>
> What we covered
> - **The voice loop** - STT -> LLM -> TTS; total delay is the sum of the parts.
> - **How STT hears** - waveform -> spectrogram -> encoder-decoder -> text (Whisper); batch vs streaming; WER.
> - **Using STT** - Whisper (batch/files), a streaming API (live), Gemini native audio (transcribe + understand).
> - **How TTS speaks** - acoustic model + vocoder -> waveform; SSML prosody; zero-shot cloning by conditioning on a reference.
> - **Voice agents** - cascaded (auditable) vs speech-to-speech (fast, opaque); LiveKit/Pipecat; the latency budget.
> - **Choosing and responsible use** - pick by constraint (WER/MOS are close); voice is biometric, so consent + watermarking ship with it.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Voice AI- Give Your AI Ears and a Voice**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_7.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.7.html` - regenerate this notebook whenever the source HTML is updated.*
